##### Selected target: churn (binary classification). Business rationale: churn is a direct revenue driver, acquiring a new subscriber costs 5–7× more than retaining one, and the dataset provides behavioral signals (usage, billing, device, complaints) that naturally precede churn. This is also a cleanly labeled and actionable target (retention campaigns can be directly targeted).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    cross_val_predict, RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve
)
from sklearn.inspection import permutation_importance
from scipy.stats import loguniform, randint, uniform

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[WARN] XGBoost is unavailable — falling back to GradientBoostingClassifier.")

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_HGB = True
except ImportError:
    HAS_HGB = False

RNG = 42
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

In [2]:
folder = Path(r"C:\FILE PROYEKAN\GCI World 2026\Final Assignment\telecom")

def find_by_stem(folder: Path, stem: str) -> Path:
    matches = [p for p in folder.iterdir() if p.is_file() and p.stem.lower() == stem.lower()]
    assert len(matches) == 1, f"Expected 1 file for {stem}, got {matches}"
    return matches[0]

def load_table(path: Path) -> pd.DataFrame:
    s = path.suffix.lower()
    if s == ".csv":  return pd.read_csv(path)
    if s in (".xlsx", ".xls"):  return pd.read_excel(path)
    try:    return pd.read_csv(path)
    except: return pd.read_excel(path)

client = load_table(find_by_stem(folder, "Client"))
record = load_table(find_by_stem(folder, "Record"))

# Validate 1:1 cardinality before merging (prevents row explosion)
df = record.merge(client, on="Customer_ID", how="inner", validate="one_to_one")
print("Merged shape:", df.shape)
print("Churn rate:", df["churn"].mean().round(4))

Merged shape: (100000, 100)
Churn rate: 0.4956


Important note: EDA code states churn ~50:50. Re-check with df['churn'].mean() - if it turns out to be heavily imbalanced (e.g., 2%), the entire evaluation strategy should be based on PR-AUC and class_weight, not accuracy. This code anticipates both cases.

In [3]:
y = df["churn"].astype(int)
X = df.drop(columns=["churn", "Customer_ID"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RNG
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")

Train: (80000, 98), Test: (20000, 98)
Churn rate train: 0.4956 | test: 0.4956


In [4]:
EPS = 1e-6

def add_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X["revenue_per_min"]       = X["totrev"] / (X["totmou"] + EPS)
    X["revenue_per_call"]      = X["totrev"] / (X["totcalls"] + EPS)
    X["avg_call_duration"]     = X["totmou"] / (X["totcalls"] + EPS)
    X["rev_per_month"]         = X["totrev"] / (X["months"] + EPS)
    X["mou_per_month"]         = X["totmou"] / (X["months"] + EPS)
    X["drop_rate"]             = X["drop_vce_Mean"] / (X["plcd_vce_Mean"] + EPS)
    X["block_rate"]            = X["blck_vce_Mean"] / (X["plcd_vce_Mean"] + EPS)
    X["unans_rate"]            = X["unan_vce_Mean"] / (X["plcd_vce_Mean"] + EPS)
    X["complete_rate"]         = X["complete_Mean"] / (X["attempt_Mean"] + EPS)
    X["custcare_per_attempt"]  = X["custcare_Mean"] / (X["attempt_Mean"] + EPS)
    X["overage_rev_ratio"]     = X["ovrrev_Mean"]   / (X["rev_Mean"] + EPS)
    X["sub_active_ratio"]      = X["actvsubs"]      / (X["uniqsubs"] + EPS)
    X["is_new_customer"]       = (X["months"] <= 6).astype(int)
    X["usage_declining"]       = (X["change_mou"] < 0).astype(int)
    X["revenue_declining"]     = (X["change_rev"] < 0).astype(int)
    X["recent_usage_trend"]    = (X["avg3mou"] - X["avg6mou"]) / (X["avg6mou"] + EPS)
    return X

X_train = add_features(X_train)
X_test  = add_features(X_test)

In [5]:
# Categorical cardinality reduction (fit only on train):
def reduce_cardinality(X_tr, X_te, cat_cols, top_k=30, min_count=100):
    X_tr, X_te = X_tr.copy(), X_te.copy()
    for col in cat_cols:
        vc = X_tr[col].value_counts()
        keep = vc[(vc >= min_count)].index[:top_k]
        X_tr[col] = X_tr[col].where(X_tr[col].isin(keep), "OTHER")
        X_te[col] = X_te[col].where(X_te[col].isin(keep), "OTHER")
    return X_tr, X_te

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
X_train, X_test = reduce_cardinality(X_train, X_test, cat_cols, top_k=30, min_count=100)
print(f"Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}")

Numeric: 93 | Categorical: 21


From EDA, ~10 columns have >10% missing (numbcars 49%, dwllsize 38%, etc.). So the strategy is:

Numeric → median + missing indicator. Median is robust to outliers. add_indicator=True lets the model learn that "customer did not report income" is itself a signal (likely a specific segment). This is critical for income, numbcars, etc.

Categorical → explicit "MISSING" category. More informative than most frequent because missingness on ownrent may correlate with churn.

In [6]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler",  StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore",
                              min_frequency=0.005,  # drop levels <0.5%
                              sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
], remainder="drop")

Primary metric: ROC-AUC (ranking-based, robust to class imbalance) for tuning. Secondary metrics: PR-AUC / average precision (more informative when churn <5%), precision, recall, F1, and confusion matrix.

Validation: Stratified 5-fold CV on the train set only. Test set is held out until final evaluation.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)

scoring = {
    "roc_auc":   "roc_auc",
    "pr_auc":    "average_precision",
    "precision": "precision",
    "recall":    "recall",
    "f1":        "f1",
}

In [8]:
pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

models = {
    "Baseline (Dummy prior)": DummyClassifier(strategy="prior", random_state=RNG),

    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", solver="liblinear", random_state=RNG
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced_subsample",
        n_jobs=-1, random_state=RNG
    ),
}

if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=pos_weight,
        eval_metric="logloss", tree_method="hist",
        n_jobs=-1, random_state=RNG
    )
elif HAS_HGB:
    models["HistGradientBoosting"] = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_depth=None,
        class_weight="balanced", random_state=RNG
    )

for name, m in models.items():
    print(f"{name}: {type(m).__name__}")

Baseline (Dummy prior): DummyClassifier
Logistic Regression: LogisticRegression
Random Forest: RandomForestClassifier
XGBoost: XGBClassifier


In [9]:
# Baseline cross-validation (before tuning):
cv_rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=scoring, n_jobs=-1,
                         return_train_score=False)
    cv_rows.append({
        "Model": name,
        "CV ROC-AUC": f"{res['test_roc_auc'].mean():.4f} ± {res['test_roc_auc'].std():.4f}",
        "CV PR-AUC":  f"{res['test_pr_auc'].mean():.4f}",
        "CV Precision": f"{res['test_precision'].mean():.4f}",
        "CV Recall":    f"{res['test_recall'].mean():.4f}",
        "CV F1":        f"{res['test_f1'].mean():.4f}",
    })

cv_table = pd.DataFrame(cv_rows).set_index("Model")
print(cv_table.to_string())

                             CV ROC-AUC CV PR-AUC CV Precision CV Recall   CV F1
Model                                                                           
Baseline (Dummy prior)  0.5000 ± 0.0000    0.4956       0.0000    0.0000  0.0000
Logistic Regression     0.6380 ± 0.0031    0.6144       0.5941    0.6062  0.6001
Random Forest           0.6754 ± 0.0049    0.6559       0.6219    0.6247  0.6233
XGBoost                 0.6965 ± 0.0051    0.6811       0.6337    0.6580  0.6456


## Hyperparameter Tuning (Measured, Not "Trial and Error")

Tuning is only for non-baseline models, using RandomizedSearchCV (more efficient than Grid for large spaces), inside the train set, with scoring = roc_auc and CV = StratifiedKFold(3). We choose roc_auc (not f1) because the final threshold will be optimized separately.

In [11]:
param_spaces = {
    "Logistic Regression": {
        "clf__C":        loguniform(1e-3, 1e2),
        "clf__penalty":  ["l2"],
    },
    "Random Forest": {
        "clf__n_estimators":     randint(200, 600),
        "clf__max_depth":        [None, 10, 20, 30],
        "clf__min_samples_leaf": randint(1, 20),
        "clf__max_features":     ["sqrt", "log2", None],
    },
    "XGBoost": {
        "clf__n_estimators":     randint(200, 600),
        "clf__max_depth":        randint(3, 10),
        "clf__learning_rate":    loguniform(0.01, 0.2),
        "clf__subsample":        uniform(0.6, 0.4),
        "clf__colsample_bytree": uniform(0.6, 0.4),
        "clf__min_child_weight": randint(1, 10),
    },
}

tuned = {}
best_params = {}
for name, space in param_spaces.items():
    if name not in models:
        continue
    pipe = Pipeline([("prep", preprocessor), ("clf", models[name])])
    search = RandomizedSearchCV(
        pipe, space, n_iter=4, cv=StratifiedKFold(3, shuffle=True, random_state=RNG),
        scoring="roc_auc", n_jobs=-1, random_state=RNG, refit=True, verbose=0
    )
    search.fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    best_params[name] = search.best_params_
    print(f"[{name}] best CV ROC-AUC = {search.best_score_:.4f}")
    print(f"          params = {search.best_params_}")

[Logistic Regression] best CV ROC-AUC = 0.6376
          params = {'clf__C': np.float64(56.69849511478853), 'clf__penalty': 'l2'}
[Random Forest] best CV ROC-AUC = 0.6798
          params = {'clf__max_depth': 20, 'clf__max_features': None, 'clf__min_samples_leaf': 11, 'clf__n_estimators': 287}
[XGBoost] best CV ROC-AUC = 0.6943
          params = {'clf__colsample_bytree': np.float64(0.996884623716487), 'clf__learning_rate': np.float64(0.06358606736176979), 'clf__max_depth': 4, 'clf__min_child_weight': 6, 'clf__n_estimators': 452, 'clf__subsample': np.float64(0.7727780074568463)}


## Threshold Optimization (Validation-Based, Not Test)

For churn, a 0.5 threshold is rarely optimal. We choose the threshold that maximizes F1 on out-of-fold predictions from train - not test. This keeps the test set truly clean.

In [12]:
from sklearn.metrics import precision_recall_curve

def optimal_threshold(model, X, y, cv):
    oof_proba = cross_val_predict(model, X, y, cv=cv,
                                  method="predict_proba", n_jobs=-1)[:, 1]
    prec, rec, thr = precision_recall_curve(y, oof_proba)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    return float(thr[f1.argmax()]), float(f1.max())

thresholds = {}
for name, model in tuned.items():
    thr, f1_cv = optimal_threshold(model, X_train, y_train, cv)
    thresholds[name] = thr
    print(f"[{name}] threshold={thr:.3f} | OOF F1={f1_cv:.4f}")

[Logistic Regression] threshold=0.358 | OOF F1=0.6701
[Random Forest] threshold=0.347 | OOF F1=0.6854
[XGBoost] threshold=0.358 | OOF F1=0.6894


## Final Evaluation on Hold-Out Test

Run only once on the test set. All numbers below are the final ones.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import pandas as pd

# 1. Fit baseline once, on train only
baseline_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf",  DummyClassifier(strategy="prior", random_state=RNG))
])
baseline_pipe.fit(X_train, y_train)

# 2. Evaluation helper (robust to all-0/all-1 predictions)
def eval_model(name, model, X_te, y_te, thr):
    proba = model.predict_proba(X_te)[:, 1]
    pred  = (proba >= thr).astype(int)
    cm    = confusion_matrix(y_te, pred, labels=[0, 1])
    return {
        "Model": name,
        "Threshold": round(thr, 3),
        "Test ROC-AUC":  round(roc_auc_score(y_te, proba), 4),
        "Test PR-AUC":   round(average_precision_score(y_te, proba), 4),
        "Test Precision": round(precision_score(y_te, pred, zero_division=0), 4),
        "Test Recall":    round(recall_score(y_te, pred, zero_division=0), 4),
        "Test F1":        round(f1_score(y_te, pred, zero_division=0), 4),
        "TN": int(cm[0, 0]), "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]), "TP": int(cm[1, 1]),
    }

# 3. Evaluate all models on held-out test
final_rows = []
final_rows.append(
    eval_model("Baseline (Dummy prior)", baseline_pipe, X_test, y_test, thr=0.5)
)
for name, model in tuned.items():
    final_rows.append(
        eval_model(name, model, X_test, y_test, thresholds[name])
    )

final_table = pd.DataFrame(final_rows).set_index("Model")
print(final_table.to_string())

                        Threshold  Test ROC-AUC  Test PR-AUC  Test Precision  Test Recall  Test F1     TN    FP    FN    TP
Model                                                                                                                      
Baseline (Dummy prior)      0.500        0.5000       0.4956          0.0000       0.0000   0.0000  10088     0  9912     0
Logistic Regression         0.358        0.6406       0.6199          0.5250       0.9254   0.6700   1790  8298   739  9173
Random Forest               0.347        0.6800       0.6689          0.5468       0.9147   0.6845   2573  7515   845  9067
XGBoost                     0.358        0.6965       0.6843          0.5583       0.8921   0.6868   3094  6994  1070  8842


## Feature Importance - What Drives Churn?

Two approaches: LogReg coefficients (interpretable, clear direction of effect) and permutation importance (model-agnostic, measures impact on ROC-AUC).

In [ ]:
def feature_names(preprocessor, num_cols, cat_cols):
    num_imp = preprocessor.named_transformers_["num"].named_steps["imputer"]
    names = num_cols.copy()
    if hasattr(num_imp, "indicator_") and num_imp.indicator_ is not None:
        names += [f"{num_cols[i]}_missing" for i in num_imp.indicator_.features_]
    ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    names += ohe.get_feature_names_out(cat_cols).tolist()
    return names

# LogReg coefficient
if "Logistic Regression" in tuned:
    lr = tuned["Logistic Regression"]
    prep = lr.named_steps["prep"]
    fn = feature_names(prep, num_cols, cat_cols)
    coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=fn)
    print("\nTop 15 features INCREASING churn (coef > 0):")
    print(coefs.sort_values(ascending=False).head(15).to_string())
    print("\nTop 15 features DECREASING churn (coef < 0):")
    print(coefs.sort_values().head(15).to_string())

# Permutation importance for tree models
for name in ["Random Forest", "XGBoost"]:
    if name not in tuned:
        continue
    model = tuned[name]
    # use a sample of X_test for speed
    sample_idx = np.random.RandomState(RNG).choice(len(X_test), size=5000, replace=False)
    pi = permutation_importance(
        model, X_test.iloc[sample_idx], y_test.iloc[sample_idx],
        n_repeats=5, scoring="roc_auc", random_state=RNG, n_jobs=-1
    )
    imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
    print(f"\n[{name}] Top 15 permutation importance (Δ ROC-AUC):")
    print(imp.head(15).to_string())


Top 15 features INCREASING churn (coef > 0):
vceovr_Mean                      2.931871
adjmou                           1.846153
totrev                           1.603921
totcalls                         1.513723
mou_opkd_Mean                    1.270344
mou_per_month                    1.033357
avgrev                           0.693504
hnd_webcap_infrequent_sklearn    0.674521
mou_pead_Mean                    0.584138
cc_mou_Mean                      0.538007
area_infrequent_sklearn          0.492984
crclscod_A2                      0.469316
rev_Mean                         0.404620
ethnic_O                         0.322566
datovr_Mean                      0.317995

Top 15 features DECREASING churn (coef < 0):
ovrrev_Mean                   -3.045566
totmou                        -1.839654
mou_cdat_Mean                 -1.633150
adjrev                        -1.588868
adjqty                        -1.553047
rev_per_month                 -0.779214
ccrndmou_Mean                 -0.66721

High eqpdays (old handset) as a risk factor → a device upgrade program for this segment could suppress churn. EDA hypothesis #1 is quantitatively confirmed/rejected.

High drop_rate, unans_rate, custcare_per_attempt → network quality and service frustration are drivers. Network improvements in high-drop_rate areas have retention ROI.

revenue_declining / negative change_rev → customers downgrading before churn. Retargeting package offers before they leave entirely.

is_new_customer is typically a top feature - churn is concentrated in the early months. Onboarding programs are the biggest lever.

area, crclscod one-hot appearing → churn is geographic/segmental; retention budget allocation can be mapped per area.

## Limitations (Must Be Communicated to the Client)

No temporal split is available. I use a random split because there is no observation_date. In production, the model is trained on period N and tested on N+1, performance may be lower (concept drift). Real validation must use time-based CV.

change_mou / change_rev are measured at the observation date, still pre-churn (safe), but business-wise they reflect "behavior before observation". If any label leaks after the 31–60 day window, these features will look extremely important. An audit of date definitions with the client's data team is required.

Median imputation assumes MCAR/MAR. If missingness on income is MNAR (customers intentionally do not report), the bias remains. Missing indicators are only a mitigation, not a solution.

Cardinality truncation (top-30). Rare crclscod levels may carry signal, a better alternative is CV-based target encoding or CatBoost. This pipeline chooses simplicity.

Class imbalance (if truly 2%) → precision will be relatively low. scale_pos_weight helps recall, but does not change the fact that 20-30% precision is typical for churn. Communicate this as a ranking tool, not an absolute classifier.

Feature importance ≠ causal. eqpdays being important ≠ "upgrading the handset will prevent churn". For causal claims, you need uplift modeling or an A/B test.

Cost optimization is not yet explicit. We optimize F1. In production, a cost matrix must be defined: FN cost = lost CLV, FP cost = retention offer cost (discount, CS call). The final threshold must be chosen on a cost curve, not F1.

The model does not consider retention team capacity. If the team can only handle 5,000 customers/month, the threshold must be set so that the top 5,000 (top-k) are allocated, not a global threshold.

In [17]:
# EXAMPLE FRAMEWORK (not final numbers — fill in after running)
n_test = len(y_test)
selected_model = "XGBoost"  # choose after seeing results
row = final_table.loc[selected_model]

TP = row["TP"]; FP = row["FP"]; FN = row["FN"]
# Business assumptions (fill in from finance/client):
CLV = 500_000          # IDR, example
cost_retention = 50_000
uplift_retention = 0.30  # 30% of customers who receive the offer stay

saved_revenue = TP * uplift_retention * CLV
offer_cost    = (TP + FP) * cost_retention
net_benefit   = saved_revenue - offer_cost
roi = net_benefit / max(offer_cost, 1)
print(f"Estimated net benefit per test window: IDR {net_benefit:,.0f} (ROI {roi:.1%})")

Estimated net benefit per test window: IDR 534,500,000 (ROI 67.5%)


The numbers above are a formula, not the result, fill CLV, uplift, and cost_retention with the client's real data. This is what turns the proposal into a business proposal, not a technical report.